# Interactive Explainability Dashboard

This notebook allows you to select any trained model and test example to inspect its LIME and SHAP interpretability explanation

### Instructions:
1. Run the cells below.
2. Select a model from the dropdown (e.g. `robert_red` for REDv2 emotions, `bert_romanian` for Romanian articles, or `distilbert` for IMDB reviews).
3. Use the slider or enter an index (0 to 48/49) to choose the test example you want to inspect.
4. View the example details, raw text, comparison plot, and interactive LIME report.

In [11]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display
import ipywidgets as widgets

# Change working directory to project root if running from notebooks directory
if Path.cwd().name == "notebooks":
    os.chdir(Path("..").resolve())

# Ensure project root and digit-prefixed folders are in the path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root / "2_Romanian") not in sys.path:
    sys.path.insert(0, str(project_root / "shared" / "migrated"))
    sys.path.insert(0, str(project_root / "2_Romanian" / "shared"))
if str(project_root / "1_English") not in sys.path:
    sys.path.insert(0, str(project_root / "1_English" / "shared"))

# Define robust results directory path
# Define decentralized pipeline results folders map
MODEL_PIPELINE_DIRS = {
    "robert_red": "2_Romanian/2_RoBERTa_Emotion_RED",
    "robert_conv": "2_Romanian/3_RoBERTa_Conv_Emotion_RED",
    "robert_ensemble": "2_Romanian/4_RoBERTa_Ensemble_Emotion_RED",
    "bert_romanian": "2_Romanian/1_BERT_Sentiment_Articles",
    "distilbert": "1_English/4_DistilBERT_SA_Practical",
    "sentencebert": "1_English/5_SentenceBERT_Practical"
}

# Set seaborn style for premium aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Helvetica"]

In [12]:
# Model metadata configurations
MODEL_CONFIGS = {
    "robert_red": {
        "dataset_type": "red",
        "text_col": "text",
        "class_names": ["sadness", "surprise", "fear", "anger", "neutral", "trust", "joy"]
    },
    "sentencebert": {
        "dataset_type": "imdb",
        "text_col": "review",
        "class_names": ["negative", "positive"]
    },
    "bert_romanian": {
        "dataset_type": "ro_articles",
        "text_col": "article",
        "class_names": ["negative", "neutral", "positive"]
    },
    "distilbert": {
        "dataset_type": "imdb",
        "text_col": "review",
        "class_names": ["negative", "positive"]
    }
}

In [13]:
# Cache loaded datasets in memory to keep the interface extremely fast
DATASETS_CACHE = {}
PREPROCESSED_CACHE = {}

def get_dataset(dataset_type):
    if dataset_type in DATASETS_CACHE:
        return DATASETS_CACHE[dataset_type]

    print(f"Loading '{dataset_type}' dataset test split...")
    if dataset_type == "red":
        from dataset_red import load_splits, EMOTION_LABELS
        _, _, df_test = load_splits(task="classification")
        # Filter to single-label texts for explanation compatibility
        agreed_cols = [f"agreed_label_{e}" for e in EMOTION_LABELS]
        df_test = df_test[df_test["num_emotions"] == 1].reset_index(drop=True)
        df_test["label"] = df_test[agreed_cols].values.argmax(axis=1)
    elif dataset_type == "ro_articles":
        from dataset_ro import load_splits
        _, _, df_test = load_splits()
    elif dataset_type == "imdb":
        from dataset_en import load_splits
        _, _, df_test = load_splits()
    else:
        raise ValueError(f"Unknown dataset_type: {dataset_type}")

    DATASETS_CACHE[dataset_type] = df_test
    return df_test

def get_preprocessed_text(dataset_type, db_idx):
    cache_key = dataset_type
    if cache_key in PREPROCESSED_CACHE:
        return PREPROCESSED_CACHE[cache_key][db_idx]

    print(f"Loading '{dataset_type}' preprocessed test texts...")
    if dataset_type == "red":
        from dataset_red import load_preprocessed
    elif dataset_type == "ro_articles":
        from dataset_ro import load_preprocessed
    elif dataset_type == "imdb":
        from dataset_en import load_preprocessed
    else:
        raise ValueError(f"Unknown dataset_type: {dataset_type}")

    preprocessed_texts = load_preprocessed("test")
    PREPROCESSED_CACHE[cache_key] = preprocessed_texts
    return preprocessed_texts[db_idx]

In [14]:
def show_explanation(selected_model, idx):
    cfg = MODEL_CONFIGS[selected_model]

    # Load LIME and SHAP summaries for the selected model run
    pipeline_dir = Path(MODEL_PIPELINE_DIRS[selected_model])
    lime_dir = pipeline_dir / "Results" / "lime_explanations"
    shap_dir = pipeline_dir / "Results" / "shap_explanations"

    try:
        with open(lime_dir / "lime_summary.json", "r", encoding="utf-8") as f:
            lime_summary = json.load(f)
        with open(shap_dir / "shap_summary.json", "r", encoding="utf-8") as f:
            shap_summary = json.load(f)
    except FileNotFoundError as e:
        display(HTML(
            f"<div style='color: red; padding: 10px; border: 1px solid red; border-radius: 4px;'>"
            f"<b>Error:</b> Explanations not found for model '{selected_model}'. "
            f"Please verify that explainability scripts have been run for this model.</div>"
        ))
        return

    if not lime_summary or not shap_summary:
        print("Explanations summary files are empty.")
        return

    # Bound check the index slider
    idx = min(max(0, idx), len(lime_summary) - 1)

    lime_entry = lime_summary[idx]
    shap_entry = shap_summary[idx]
    db_idx = lime_entry["example_idx"]

    # Load original raw text and preprocessed text
    try:
        df_test = get_dataset(cfg["dataset_type"])
        raw_text = df_test.loc[db_idx, cfg["text_col"]]
        prep_text = get_preprocessed_text(cfg["dataset_type"], db_idx)
    except Exception as e:
        raw_text = f"[Error loading raw text from dataset: {e}]"
        prep_text = f"[Error loading preprocessed text: {e}]"

    # Correct prediction indicator
    correct_status = (
        "<span style='color: #2ecc71; font-weight: bold;'>\u2705 Correct</span>"
        if lime_entry["correct"]
        else "<span style='color: #e74c3c; font-weight: bold;'>\u274c Incorrect</span>"
    )

    # 1. Styled Metadata Box
    meta_html = f"""
    <div style="background-color: #f8f9fa; border-left: 6px solid #3498db; padding: 15px; margin-bottom: 15px; border-radius: 6px;">
        <h3 style="margin: 0 0 10px 0; color: #2c3e50; font-size: 1.25em;">Example #{idx} (Dataset Index: {db_idx})</h3>
        <div style="display: grid; grid-template-columns: 1fr 1fr 1fr; gap: 15px;">
            <div><b>True Label:</b> <span style="text-transform: capitalize; color: #34495e;">{lime_entry['true_label']}</span></div>
            <div><b>Predicted Label:</b> <span style="text-transform: capitalize; color: #34495e;">{lime_entry['predicted_label']}</span></div>
            <div><b>Status:</b> {correct_status}</div>
        </div>
    </div>
    """
    display(HTML(meta_html))

    # 2. Collapsible Raw Text (Dropdown)
    raw_text_html = f"""
    <details style="margin-bottom: 15px; border: 1px solid #dee2e6; border-radius: 8px; background-color: #f8f9fa; padding: 12px; font-family: system-ui, sans-serif;">
        <summary style="font-size: 0.85em; font-weight: bold; color: #495057; cursor: pointer; text-transform: uppercase; outline: none; display: list-item;">
            View Original Raw Text ({cfg['text_col']})
        </summary>
        <p style="margin-top: 10px; font-size: 0.95em; line-height: 1.6; color: #2c3e50; font-family: Georgia, serif; white-space: pre-wrap;">{raw_text}</p>
    </details>
    """
    display(HTML(raw_text_html))

    # 3. Highlighted Preprocessed Text
    lime_top = lime_entry["top_words"][:10]
    lime_dict = {w.lower(): wt for w, wt in lime_top}

    # Tokenize preprocessed text by splitting on spaces
    words = prep_text.split()
    highlighted_words = []
    for word in words:
        clean_word = word.lower().strip(".,!?\"'()[]{}")
        weight = lime_dict.get(clean_word, 0)
        if weight > 0.005:
            highlighted_words.append(
                f'<span style="background-color: rgba(46, 204, 113, 0.22); padding: 2px 4px; '
                f'border-radius: 4px; font-weight: 500; border-bottom: 2px solid #2ecc71;">{word}</span>'
            )
        elif weight < -0.005:
            highlighted_words.append(
                f'<span style="background-color: rgba(231, 76, 60, 0.22); padding: 2px 4px; '
                f'border-radius: 4px; font-weight: 500; border-bottom: 2px solid #e74c3c;">{word}</span>'
            )
            highlighted_words.append(word)
    highlighted_html = " ".join(highlighted_words)

    prep_text_html = f"""
    <details style="margin-bottom: 20px; border: 1px solid #dee2e6; border-radius: 8px; background-color: #ffffff; padding: 12px;">
        <summary style="font-family: system-ui, sans-serif; font-size: 0.85em; font-weight: bold;
                        color: #495057; cursor: pointer; text-transform: uppercase; outline: none;
                        display: list-item;">
            View Preprocessed Text (with LIME highlights)
        </summary>

        <div style="margin-top: 15px; font-size: 1.05em; line-height: 1.8;
                    box-shadow: 0 1px 3px rgba(0,0,0,.05);">
            <p style="font-family: Georgia, serif; color: #212529; margin: 0;">
                {highlighted_html}
            </p>

            <div style="margin-top: 15px; font-family: system-ui, sans-serif;
                        font-size: 0.8em; color: #6c757d; display: flex; gap: 15px;">
                <span style="display: flex; align-items: center; gap: 5px;">
                    <span style="display: inline-block; width: 12px; height: 12px;
                                background-color: rgba(46, 204, 113, 0.25);
                                border-bottom: 2px solid #2ecc71; border-radius: 2px;"></span>
                    Positive Impact
                </span>

                <span style="display: flex; align-items: center; gap: 5px;">
                    <span style="display: inline-block; width: 12px; height: 12px;
                                background-color: rgba(231, 76, 60, 0.25);
                                border-bottom: 2px solid #e74c3c; border-radius: 2px;"></span>
                    Negative Impact
                </span>
            </div>
        </div>
    </details>
    """
    display(HTML(prep_text_html))

    # 4. Plot LIME and SHAP top words side-by-side
    lime_words = [w for w, _ in lime_top]
    lime_weights = [v for _, v in lime_top]

    shap_top = shap_entry["top_tokens"][:10]
    shap_tokens = [t for t, _ in shap_top]
    shap_weights = [v for _, v in shap_top]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # LIME chart
    l_colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in lime_weights]
    axes[0].barh(lime_words[::-1], lime_weights[::-1], color=l_colors[::-1], edgecolor="black", height=0.6)
    axes[0].axvline(0, color="black", linewidth=0.8, linestyle="--")
    axes[0].set_xlabel("LIME Coefficient (Weight)", fontsize=11)
    axes[0].set_title("LIME Explainer (Top 10 features)", fontsize=12, fontweight="bold", pad=10)
    axes[0].tick_params(labelsize=10.5)

    # SHAP chart
    s_colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in shap_weights]
    axes[1].barh(shap_tokens[::-1], shap_weights[::-1], color=s_colors[::-1], edgecolor="black", height=0.6)
    axes[1].axvline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_xlabel("SHAP Value (Impact)", fontsize=11)
    axes[1].set_title("SHAP Explainer (Top 10 features)", fontsize=12, fontweight="bold", pad=10)
    axes[1].tick_params(labelsize=10.5)

    plt.suptitle("Feature Importances Comparison (LIME vs SHAP)", fontsize=14, fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.show()

    # 5. Interactive LIME HTML Report
    lime_html_path = lime_dir / lime_entry["html_file"]
    if lime_html_path.exists():
        display(HTML("<hr style='border-top: 1px solid #dee2e6; margin: 30px 0 20px 0;'>"))
        display(HTML("<h4 style='font-family: system-ui, sans-serif; color: #2c3e50; margin-bottom: 15px;'>Interactive LIME Visual Report</h4>"))
        display(HTML(lime_html_path.read_text(encoding="utf-8")))
    else:
        display(HTML(f"<div style='color: orange; margin-top: 20px;'><b>Note:</b> LIME HTML file not found at '{lime_html_path}'</div>"))

In [15]:
# Set up the dropdown list of models dynamically from results folder
results_base = None
available_models = []
for model_name, p_dir in MODEL_PIPELINE_DIRS.items():
    summary_path = Path(p_dir) / "Results" / "lime_explanations" / "lime_summary.json"
    if summary_path.exists():
        available_models.append(model_name)
available_models = sorted(available_models)
if not available_models:
    available_models = sorted(list(MODEL_PIPELINE_DIRS.keys()))

print("Available models found:", available_models)

model_dropdown = widgets.Dropdown(
    options=available_models,
    value=available_models[0] if available_models else None,
    description="Model Run:",
    disabled=False,
)

# The slider to select the example index (updated dynamically)
idx_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=48,  # Default, will update dynamically
    step=1,
    description="Example:",
    continuous_update=False,
    orientation="horizontal",
    readout=True,
)

# The text box to select the example index directly
idx_input = widgets.BoundedIntText(
    value=0,
    min=0,
    max=48,
    step=1,
    description="Index Input:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="140px"),
)

# Link the slider and text box bidirectionally
widgets.jslink((idx_slider, "value"), (idx_input, "value"))

# Dynamic slider range update on model change
def on_model_change(change):
    selected_model = change["new"]
    pipeline_dir = Path(MODEL_PIPELINE_DIRS[selected_model])
    summary_path = pipeline_dir / "Results" / "lime_explanations" / "lime_summary.json"
    if summary_path.exists():
        try:
            with open(summary_path, "r", encoding="utf-8") as f:
                summary_entries = json.load(f)
            if summary_entries:
                max_val = len(summary_entries) - 1
                idx_slider.max = max_val
                idx_input.max = max_val
                idx_slider.value = 0
                idx_input.value = 0
        except Exception:
            pass

model_dropdown.observe(on_model_change, names="value")

# Create the layout grid
ui = widgets.VBox([
    widgets.HTML("<h2 style='font-family: sans-serif; color: #2c3e50; margin-bottom: 15px;'>Select Explanation Run:</h2>"),
    widgets.HBox([model_dropdown, idx_slider, idx_input])
])
out = widgets.interactive_output(show_explanation, {"selected_model": model_dropdown, "idx": idx_slider})

# Trigger initial config update
if model_dropdown.value:
    on_model_change({"new": model_dropdown.value})

display(ui, out)

Available models found: ['bert_romanian', 'distilbert', 'robert_conv', 'robert_ensemble', 'robert_red', 'sentencebert']


Output()